# 03 · Vectores del ciclo de vida: empleo, emparejamiento y educación

Estima, sobre el panel de la EPH continua (3.er trimestre, 2016-2024), las probabilidades que
gobiernan la cascada anual del motor de microsimulación, salvo fecundidad y mortalidad:

| Vector | Evento | Universo | Clave de la grilla |
|---|---|---|---|
| `employment_vector` | ocupada | mujeres 15-50 | provincia × tercil × edad × educación |
| `partner_vector` | en pareja, pareja ocupada, clima educativo del hogar (5 clases) | mujeres 15-50 | ídem |
| `education_stock_vector` | nivel educativo máximo (7 clases) | personas 25+ | sexo × provincia × tercil × clima × **cohorte** |
| `education_flow_vector` | asiste a un establecimiento educativo | personas 3-80 | sexo × provincia × tercil × edad × clima × destino educativo |

**Arquitectura de cada vector**

- **Perfil `f(X)`:** XGBoost calibrado isotónicamente, sin el año entre sus variables. La
  validación usa el holdout temporal 2022-2024, y tanto la calibración como los remuestreos se
  agrupan por vivienda (`codusu`), porque la EPH es un panel rotativo.
- **Drift secular:** pendiente en logit por estrato (tramo etario × nivel educativo; etapa escolar
  × clima educativo para el flujo; sexo × clima educativo para el stock), estimada con un GLM
  binomial individual con errores estándar *cluster* por vivienda, excluyendo 2020-2021, y
  encogida hacia la media con el estimador de DerSimonian-Laird. Se proyecta con una forma
  saturante `β·τ·(1 - e^{-Δt/τ})` con τ = 15 años y cotas lógicas por vector.
- **Stock educativo:** deriva de **cohorte de nacimiento** (lineal hacia atrás, saturante hacia
  adelante) desde la cohorte ancla 1999, para cubrir las cohortes 1925-2075.
- **Flujo educativo condicionado al destino:** el destino de quien todavía estudia se imputa
  con el modelo de stock, restringido a niveles ≥ al ya alcanzado.

**Entradas:** `datos/eph/personas_tot.urb_3T_2016.txt` … `_2025.txt`.

**Salidas:**
- `resultados/intermedios/vectores/{employment,partner,education_stock,education_flow}_vector.parquet`
- `resultados/intermedios/diagnosticos/drift_vectores.csv` (heterogeneidad del *drift*, Apéndice)
- `resultados/intermedios/diagnosticos/metricas_vectores.csv` (validación de los modelos, Apéndice)

**Tiempo aproximado:** 2 minutos.

In [1]:
import itertools
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import xgboost as xgb
from scipy import stats
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
DIR_VEC = Path("resultados/intermedios/vectores")
DIR_DIAG = Path("resultados/intermedios/diagnosticos")
for d in (DIR_VEC, DIR_DIAG):
    d.mkdir(parents=True, exist_ok=True)

ANIO_INICIO_PANEL, ANIO_FIN_PANEL = 2016, 2024
ANIOS_EXCLUIDOS_CALENDARIO = (2020, 2021)
ANIO_CENTRADO_DRIFT = 2020
TAU_SATURACION = 15.0
N_ESTRATOS_STATUS = 3
MIN_DONANTES_IMPUTACION = 10
SEMILLA_IMPUTACION = 20260905

TRAMOS_DRIFT = [(15, 24), (25, 34), (35, 50)]
TRAMOS_ESCOLARES = [(3, 5), (6, 11), (12, 17), (18, 24), (25, 80)]
BUCKET_EDU = {0: 'Bajo (sin instr./primario)', 1: 'Bajo (sin instr./primario)',
              2: 'Bajo (sin instr./primario)', 3: 'Medio (secundario)',
              4: 'Medio (secundario)', 5: 'Alto (superior)', 6: 'Alto (superior)'}
BUCKET_CLIMA = {0: 'Bajo (sin instr./primario)', 1: 'Bajo (sin instr./primario)',
                2: 'Medio (secundario)', 3: 'Alto (superior)', 4: 'Alto (superior)'}

COHORTE_ANCLA_STOCK, COHORTE_MIN_VECTOR, COHORTE_MAX_VECTOR = 1999, 1925, 2075
COTAS_PROB = {'work': (0.010, 0.970), 'couple': (0.010, 0.950), 'swork': (0.010, 0.970),
              'cedu': (0.001, 0.990), 'inschool': (0.002, 0.995), 'max_level': (0.001, 0.990)}

aglomerado_mapeo = {32: 1, 33: 2, 2: 2, 3: 2, 34: 2, 38: 2, 40: 2, 18: 3, 61: 3, 19: 4, 48: 4,
                    22: 5, 41: 5, 23: 6, 55: 6, 25: 7, 50: 7, 29: 8, 62: 8, 7: 9, 52: 9,
                    8: 10, 44: 10, 12: 11, 43: 11, 15: 12, 47: 12, 10: 13, 51: 13, 26: 14, 57: 14,
                    27: 15, 56: 15, 4: 16, 5: 16, 60: 16, 6: 17, 14: 17, 46: 17, 13: 18, 36: 18,
                    42: 18, 30: 19, 49: 19, 9: 20, 91: 20, 45: 20, 17: 21, 53: 21, 20: 22, 58: 22,
                    31: 23, 93: 24, 54: 24}

## Funciones

In [2]:
def assign_tramo_drift(age):
    bins = [t[0] for t in TRAMOS_DRIFT] + [TRAMOS_DRIFT[-1][1] + 1]
    return pd.cut(age, bins=bins, labels=[f'{lo}-{hi}' for lo, hi in TRAMOS_DRIFT],
                  right=False, include_lowest=True)


def assign_tramo_escolar(age):
    bins = [t[0] for t in TRAMOS_ESCOLARES] + [TRAMOS_ESCOLARES[-1][1] + 1]
    return pd.cut(age, bins=bins, labels=[f'{lo}-{hi}' for lo, hi in TRAMOS_ESCOLARES],
                  right=False, include_lowest=True)


def get_ingreso_adulto(df, col_hogar='hogar_id', col_edad='ch06'):
    itf = pd.to_numeric(df['itf'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
    adultos = df.groupby(col_hogar)[col_edad].transform(lambda s: (s >= 18).sum()).clip(lower=1)
    return itf / adultos


def get_weighted_quantiles(df, value_col, weight_col, k=N_ESTRATOS_STATUS):
    valid = df[value_col].notna() & df[weight_col].notna() & (df[weight_col] > 0)
    df_valid = df[valid].sort_values(value_col, kind='mergesort')
    cum = df_valid[weight_col].cumsum()
    q = pd.cut(cum / cum.iloc[-1], bins=np.linspace(0, 1, k + 1),
               labels=range(1, k + 1), include_lowest=True)
    return q.astype('Int64').reindex(df.index)


def imputar_status(df, seed=SEMILLA_IMPUTACION, k=N_ESTRATOS_STATUS, min_donantes=MIN_DONANTES_IMPUTACION):
    rng = np.random.default_rng(seed)
    st = pd.to_numeric(df['status'], errors='coerce').to_numpy(dtype=float)
    w = pd.to_numeric(df['pondih'], errors='coerce').fillna(0.0).to_numpy(dtype=float)
    for keys in [['education', 'sex', 'clima_educativo', 'provincia'], ['education', 'sex'], None]:
        if not np.isnan(st).any():
            break
        grupos = ({None: np.arange(len(df))} if keys is None
                  else df.groupby(keys, dropna=False, observed=True).indices)
        for pos in grupos.values():
            pos = np.asarray(pos)
            falta = np.isnan(st[pos])
            if not falta.any():
                continue
            don = pos[~falta]
            if len(don) < min_donantes:
                continue
            pw = w[don]
            if not np.isfinite(pw).all() or pw.sum() <= 0:
                pw = np.ones(len(don))
            st[pos[falta]] = rng.choice(st[don], size=int(falta.sum()), replace=True, p=pw / pw.sum())
    resto = np.isnan(st)
    if resto.any():
        st[resto] = rng.integers(1, k + 1, size=int(resto.sum()))
    return pd.Series(st, index=df.index).round().astype(int)


def get_ed_years(df):
    nivel, finalizado, aprobado, educacion = df['ch12'], df['ch13'], df['ch14'], df['education']
    primaria, egb, secundaria, terciario, universidad, posgrado = 6, 9, 12, 14, 16, 18
    aprobado_valido = aprobado.notna() & ~aprobado.isin([98, 99])
    aprobado_safe = aprobado.fillna(0)
    fin1 = finalizado == 1
    val1 = pd.Series(0.0, index=df.index)
    val2 = np.select([fin1, aprobado_valido, educacion == 2],
                     [primaria, np.minimum(aprobado_safe, primaria - 0.5), primaria], default=primaria / 2)
    val3 = np.select([fin1, aprobado_valido, educacion == 2],
                     [egb, np.minimum(aprobado_safe, egb - 0.5), primaria], default=egb / 2)
    val4 = np.select([fin1, aprobado_valido, educacion == 4],
                     [secundaria, np.minimum(primaria + aprobado_safe, secundaria - 0.5), secundaria],
                     default=primaria + 3)
    val5 = np.select([fin1, aprobado_valido, educacion == 4],
                     [secundaria, np.minimum(egb + aprobado_safe, secundaria - 0.5), secundaria],
                     default=egb + 1.5)
    val6 = np.select([fin1, aprobado_valido, educacion == 6],
                     [terciario, np.minimum(secundaria + aprobado_safe, terciario - 0.5), terciario],
                     default=secundaria + 1)
    val7 = np.select([fin1, aprobado_valido, educacion == 6],
                     [universidad, np.minimum(secundaria + aprobado_safe, universidad - 0.5), universidad],
                     default=secundaria + 2.5)
    val8 = np.select([fin1, aprobado_valido],
                     [posgrado, np.minimum(universidad + aprobado_safe, posgrado - 0.5)], default=universidad + 1)
    val9 = np.select([educacion == i for i in range(1, 7)],
                     [primaria / 2, primaria, primaria + 3, secundaria, secundaria + 2, universidad], default=0.0)
    condiciones = [nivel == 1, nivel == 2, nivel == 3, nivel == 4, nivel == 5, nivel == 6, nivel == 7,
                   nivel == 8, nivel.isin([9, 99]) | nivel.isna()]
    return np.select(condiciones, [val1, val2, val3, val4, val5, val6, val7, val8, val9], default=0.0)


def assign_cedu(average_years):
    return pd.cut(average_years, bins=[-np.inf, 7, 11, 14, 16, np.inf], labels=[0, 1, 2, 3, 4], right=False)


def logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return np.log(p / (1 - p))


def expit(x):
    return 1 / (1 + np.exp(-np.asarray(x, dtype=float)))

In [3]:
def evaluar_modelo(df, features, target, weight_col='pondera', year_col='year', year_cutoff=2021,
                   num_class=None, cv_folds=5, group_col='codusu'):
    """XGBoost calibrado (isotónico, folds agrupados por vivienda). Devuelve el modelo final
    entrenado con todo el panel y las métricas del holdout temporal 2022-2024."""
    assert 'year' not in features
    df_clean = df.dropna(subset=features + [target, weight_col]).copy()
    class_map_inv = None
    if num_class:
        clases = sorted(df_clean[target].unique())
        class_map = {v: i for i, v in enumerate(clases)}
        class_map_inv = {i: v for v, i in class_map.items()}
        df_clean['_target'] = df_clean[target].map(class_map)
        y_col, n_clases = '_target', len(clases)
        params = dict(random_state=42, objective='multi:softprob', num_class=n_clases)
    else:
        y_col, n_clases = target, 2
        params = dict(random_state=42, objective='binary:logistic', eval_metric='logloss')

    def _fit(X, y, w, g):
        cv_calib = min(cv_folds, pd.Series(y).value_counts().min(), pd.Series(g).nunique())
        base = xgb.XGBClassifier(**params)
        if cv_calib >= 2:
            folds = list(GroupKFold(n_splits=cv_calib).split(X, y, groups=g))
            modelo = CalibratedClassifierCV(estimator=base, method="isotonic", cv=folds)
        else:
            modelo = base
        return modelo.fit(X, y, sample_weight=w)

    tr = df_clean[year_col] <= year_cutoff
    te = df_clean[year_col] > year_cutoff
    modelo_holdout = _fit(df_clean.loc[tr, features], df_clean.loc[tr, y_col],
                          df_clean.loc[tr, weight_col], df_clean.loc[tr, group_col])
    X_te, y_te, w_te = df_clean.loc[te, features], df_clean.loc[te, y_col], df_clean.loc[te, weight_col]

    metricas = {'target': target, 'tipo': 'multiclase' if num_class else 'binario',
                'n_clases': n_clases, 'n_train': int(tr.sum()), 'n_test': int(te.sum())}
    if num_class:
        proba = modelo_holdout.predict_proba(X_te)
        onehot = np.zeros_like(proba)
        onehot[np.arange(len(y_te)), y_te.values.astype(int)] = 1.0
        metricas.update(tasa_base=np.nan, roc_auc=np.nan, pr_auc=np.nan,
                        brier=np.average(((proba - onehot) ** 2).sum(axis=1), weights=w_te),
                        log_loss=log_loss(y_te, proba, sample_weight=w_te, labels=list(range(n_clases))))
    else:
        proba = modelo_holdout.predict_proba(X_te)[:, 1]
        metricas.update(tasa_base=float(np.average(y_te, weights=w_te)),
                        roc_auc=roc_auc_score(y_te, proba, sample_weight=w_te),
                        pr_auc=average_precision_score(y_te, proba, sample_weight=w_te),
                        brier=brier_score_loss(y_te, proba, sample_weight=w_te),
                        log_loss=log_loss(y_te, proba, sample_weight=w_te, labels=[0, 1]))
    print(f"[{target}] holdout 2022-2024: " + (f"ROC-AUC {metricas['roc_auc']:.4f}" if not num_class
                                                else f"log-loss {metricas['log_loss']:.4f}"))
    modelo_final = _fit(df_clean[features], df_clean[y_col], df_clean[weight_col], df_clean[group_col])
    return modelo_final, metricas, class_map_inv


def predecir_multiclase(modelo, X, class_map_inv, n_clases, prefijo, destino):
    probs = modelo.predict_proba(X)
    idx_de_clase = {v: k for k, v in class_map_inv.items()}
    for c in range(n_clases):
        j = idx_de_clase.get(c)
        destino[f'{prefijo}{c}'] = probs[:, j] if j is not None else 0.0

In [4]:
def _fit_logit_trend_individual(g, target, weight_col='pondera', year_col='year', group_id='codusu',
                                anio_centrado=ANIO_CENTRADO_DRIFT, min_obs=500):
    n_anios = int(g[year_col].nunique())
    y = g[target].to_numpy(dtype=float)
    if len(g) < min_obs or n_anios < 3 or y.min() == y.max():
        return np.nan, np.inf, np.nan, n_anios
    X = sm.add_constant(pd.DataFrame({'t': (g[year_col] - anio_centrado).astype(float)}, index=g.index))
    w = g[weight_col].to_numpy(dtype=float)
    w = w / w.mean()
    try:
        ajuste = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=w).fit(
            cov_type='cluster', cov_kwds={'groups': g[group_id].to_numpy()})
        beta, se = float(ajuste.params['t']), float(ajuste.bse['t'])
        if not np.isfinite(beta) or not np.isfinite(se) or se <= 0:
            return np.nan, np.inf, np.nan, n_anios
        return beta, se ** 2, float(2 * (1 - stats.norm.cdf(abs(beta / se)))), n_anios
    except Exception:
        return np.nan, np.inf, np.nan, n_anios


def compute_secular_drift(df, target, group_col, year_col='year', weight_col='pondera',
                          group_id='codusu', year_min=None, year_max=None, min_obs=500):
    """Pendiente en logit por estrato con encogimiento empírico-bayesiano (DerSimonian-Laird)."""
    cols = [group_col] if isinstance(group_col, str) else list(group_col)
    d = df.copy()
    if year_min is not None:
        d = d[d[year_col] >= year_min]
    if year_max is not None:
        d = d[d[year_col] <= year_max]
    d = d[~d[year_col].isin(ANIOS_EXCLUIDOS_CALENDARIO)]
    d = d.dropna(subset=[target, weight_col, year_col, group_id] + cols)

    filas = []
    for grupo, g in d.groupby(cols, observed=True):
        beta, var, p, n_anios = _fit_logit_trend_individual(g, target, weight_col, year_col, group_id,
                                                            min_obs=min_obs)
        clave = dict(zip(cols, grupo if isinstance(grupo, tuple) else (grupo,)))
        filas.append({**clave, 'slope': beta, 'var_slope': var, 'p_value': p,
                      'n_anios': n_anios, 'n_obs': len(g)})
    trend = pd.DataFrame(filas)

    beta_glob, _, _, _ = _fit_logit_trend_individual(d, target, weight_col, year_col, group_id, min_obs=min_obs)
    slope_global = beta_glob if np.isfinite(beta_glob) else 0.0
    s = trend['slope'].to_numpy(dtype=float)
    v = trend['var_slope'].to_numpy(dtype=float)
    valido = np.isfinite(s) & np.isfinite(v) & (v > 0)
    tau2, mu_re, k = 0.0, slope_global, int(valido.sum())
    if k >= 2:
        s_v, v_v = s[valido], v[valido]
        w = 1.0 / v_v
        Q = (w * (s_v - (w * s_v).sum() / w.sum()) ** 2).sum()
        denom = w.sum() - (w ** 2).sum() / w.sum()
        tau2 = max(0.0, (Q - (k - 1)) / denom) if denom > 0 else 0.0
        w_re = 1.0 / (tau2 + v_v)
        mu_re = (w_re * s_v).sum() / w_re.sum()
    peso = np.where(valido, tau2 / (tau2 + np.where(valido, v, 1.0)), 0.0)
    trend['slope_global'] = slope_global
    trend['tau2'] = tau2
    trend['mu_efectos_aleatorios'] = mu_re
    trend['peso_propio'] = peso
    trend['slope_final'] = peso * np.where(valido, s, 0.0) + (1.0 - peso) * mu_re
    trend['categoria'] = trend[cols].astype(str).agg(' · '.join, axis=1)
    print(f"[drift:{target}] tau^2 {tau2:.3e} | peso propio medio {100 * peso.mean():.0f}%")
    return trend


def aplicar_drift_secular_a_grid(grid, prob_base_col, group_col, drift_df,
                                 year_base=2025, year_ini=2025, year_fin=2075, tau=TAU_SATURACION):
    cols = [group_col] if isinstance(group_col, str) else list(group_col)
    n0 = len(grid)
    grid = grid.merge(drift_df[cols + ['slope_final']], on=cols, how='left')
    assert len(grid) == n0
    grid['slope_final'] = grid['slope_final'].fillna(drift_df['slope_final'].mean())
    sufijo = prob_base_col.replace('prob_', '')
    p_min, p_max = COTAS_PROB.get(''.join(ch for ch in sufijo if not ch.isdigit()), (0.001, 0.999))
    logit_base = logit(np.clip(grid[prob_base_col].values, p_min, p_max))
    years = np.arange(year_ini, year_fin + 1)
    factor_sat = tau * (1.0 - np.exp(-(years - year_base).astype(float) / tau))
    prob_proj = np.clip(expit(logit_base[:, None] + grid['slope_final'].values[:, None] * factor_sat[None, :]),
                        p_min, p_max)
    proj_df = pd.DataFrame({f'prob_{sufijo}_{y}': prob_proj[:, i] for i, y in enumerate(years)}, index=grid.index)
    return pd.concat([grid.drop(columns=['slope_final']), proj_df], axis=1)


def renormalizar_multiclase(df, prefijo, clases, year_ini=2025, year_fin=2075):
    for y in range(year_ini, year_fin + 1):
        cols = [c for c in (f'{prefijo}{c}_{y}' for c in clases) if c in df.columns]
        if cols:
            suma = df[cols].sum(axis=1).replace(0, np.nan)
            df[cols] = df[cols].div(suma, axis=0).fillna(1.0 / len(cols))
    return df

## Panel EPH 2016-2024

In [5]:
people_lists = []
for anio in range(2016, 2026):
    df = pd.read_csv(f'datos/eph/personas_tot.urb_3T_{anio}.txt', sep=';', low_memory=False)
    df.columns = df.columns.str.lower()
    df['ch06'] = df['ch06'].astype(int).replace(-1, 0)
    df['year'] = df['ano4'].astype(int)
    df['education'] = df['nivel_ed'].astype(int).replace(7, 0)
    df['sex'] = df['ch04'].map({2: 1, 1: 0})
    for c in ['ch10', 'ch12', 'ch13', 'ch14']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.rename(columns={'ch06': 'age'})
    df['provincia'] = df['aglomerado'].map(aglomerado_mapeo).fillna(0).astype(int)
    df['hogar_id'] = df['codusu'] + '_' + df['nro_hogar'].astype(str) + '_' + df['ano4'].astype(str)
    df = df.merge(df.groupby('hogar_id').size().reset_index(name='hh_size'), on='hogar_id', how='left')
    df['ed_years'] = get_ed_years(df)

    df['target_work'] = (df['estado'] == 1).astype(int)
    df['target_couple'] = df['ch07'].isin([1, 2]).astype(int)
    jefes = df[df['ch03'] == 1][['hogar_id', 'target_work', 'ed_years']].rename(
        columns={'target_work': 'j_work', 'ed_years': 'j_edu'})
    conyuges = df[df['ch03'] == 2][['hogar_id', 'target_work', 'ed_years']].rename(
        columns={'target_work': 'c_work', 'ed_years': 'c_edu'})
    df = df.merge(pd.merge(jefes, conyuges, on='hogar_id', how='outer'), on='hogar_id', how='left')
    clima_hogar = df[df['age'] >= 18].groupby('hogar_id')['ed_years'].mean().reset_index(name='average_years')
    df = df.merge(clima_hogar, on='hogar_id', how='left')

    # empleo de la pareja: sólo observable si la pareja está identificada en el hogar (si no, NaN)
    df['target_swork'] = pd.to_numeric(np.where(df['ch03'] == 1, df['c_work'],
                                                np.where(df['ch03'] == 2, df['j_work'], np.nan)), errors='coerce')
    df['target_cedu'] = assign_cedu(df['average_years']).astype('float').fillna(0).astype(int)
    df['target_inschool'] = (df['ch10'] == 1).astype(int)
    df['clima_educativo'] = df['target_cedu']

    df['ipcf'] = pd.to_numeric(df['ipcf'], errors='coerce')
    df['ingreso_adulto'] = get_ingreso_adulto(df, col_edad='age')
    df['status'] = get_weighted_quantiles(df, 'ingreso_adulto', 'pondih')
    df['status'] = imputar_status(df)
    people_lists.append(df[['codusu', 'year', 'age', 'sex', 'provincia', 'education', 'status', 'pondera',
                            'target_work', 'target_couple', 'target_swork', 'target_cedu',
                            'target_inschool', 'clima_educativo']])

people = pd.concat(people_lists, ignore_index=True)
people['codusu'] = people['codusu'].astype(str)
people['tramo_drift'] = assign_tramo_drift(people['age'])
people['tramo_escolar'] = assign_tramo_escolar(people['age'])
people['education_bucket'] = people['education'].map(BUCKET_EDU)
people['clima_bucket'] = people['clima_educativo'].map(BUCKET_CLIMA)
people = people[people['year'].between(ANIO_INICIO_PANEL, ANIO_FIN_PANEL)].reset_index(drop=True)
print(f"[panel] {len(people):,} personas, {people['codusu'].nunique():,} viviendas")

[panel] 739,880 personas, 153,986 viviendas


## Vector de empleo

In [6]:
ESTRATOS_DRIFT = ['tramo_drift', 'education_bucket']
features_mujeres = ['provincia', 'status', 'age', 'education']
mujeres_fertiles = people[(people['sex'] == 1) & (people['age'].between(15, 50))].copy()

modelo_work, metrics_work, _ = evaluar_modelo(mujeres_fertiles, features_mujeres, 'target_work')

grilla = list(itertools.product(range(1, 25), range(1, N_ESTRATOS_STATUS + 1), range(15, 51), range(0, 7)))
employment_vector = pd.DataFrame(grilla, columns=features_mujeres)
employment_vector['sex'] = 1
employment_vector['tramo_drift'] = assign_tramo_drift(employment_vector['age'])
employment_vector['education_bucket'] = employment_vector['education'].map(BUCKET_EDU)
employment_vector['prob_work'] = modelo_work.predict_proba(employment_vector[features_mujeres])[:, 1]

drift_work = compute_secular_drift(mujeres_fertiles, 'target_work', ESTRATOS_DRIFT,
                                   year_min=ANIO_INICIO_PANEL, year_max=ANIO_FIN_PANEL)
employment_vector = aplicar_drift_secular_a_grid(employment_vector, 'prob_work', ESTRATOS_DRIFT, drift_work)
employment_vector.to_parquet(DIR_VEC / 'employment_vector.parquet', engine='pyarrow', compression='snappy', index=False)

[target_work] holdout 2022-2024: ROC-AUC 0.7943


[drift:target_work] tau^2 0.000e+00 | peso propio medio 0%


## Vector de emparejamiento: en pareja, pareja ocupada y clima educativo del hogar

In [7]:
grilla = list(itertools.product([1], range(1, 25), range(1, N_ESTRATOS_STATUS + 1), range(15, 51), range(0, 7)))
partner_vector = pd.DataFrame(grilla, columns=['sex', 'provincia', 'status', 'age', 'education'])
partner_vector['tramo_drift'] = assign_tramo_drift(partner_vector['age'])
partner_vector['education_bucket'] = partner_vector['education'].map(BUCKET_EDU)

modelo_couple, metrics_couple, _ = evaluar_modelo(mujeres_fertiles, features_mujeres, 'target_couple')
df_en_pareja = mujeres_fertiles[mujeres_fertiles['target_couple'] == 1].copy()
modelo_swork, metrics_swork, _ = evaluar_modelo(df_en_pareja, features_mujeres, 'target_swork')
modelo_cedu, metrics_cedu, inv_cedu = evaluar_modelo(mujeres_fertiles, features_mujeres, 'target_cedu', num_class=5)

partner_vector['prob_couple'] = modelo_couple.predict_proba(partner_vector[features_mujeres])[:, 1]
partner_vector['prob_swork'] = modelo_swork.predict_proba(partner_vector[features_mujeres])[:, 1]
predecir_multiclase(modelo_cedu, partner_vector[features_mujeres], inv_cedu, 5, 'prob_cedu', partner_vector)

drift_couple = compute_secular_drift(mujeres_fertiles, 'target_couple', ESTRATOS_DRIFT,
                                     year_min=ANIO_INICIO_PANEL, year_max=ANIO_FIN_PANEL)
partner_vector = aplicar_drift_secular_a_grid(partner_vector, 'prob_couple', ESTRATOS_DRIFT, drift_couple)
drift_swork = compute_secular_drift(df_en_pareja, 'target_swork', ESTRATOS_DRIFT,
                                    year_min=ANIO_INICIO_PANEL, year_max=ANIO_FIN_PANEL)
partner_vector = aplicar_drift_secular_a_grid(partner_vector, 'prob_swork', ESTRATOS_DRIFT, drift_swork)

drift_cedu_por_clase = {}
for clase in range(5):
    mujeres_fertiles[f'_cedu_es_{clase}'] = (mujeres_fertiles['target_cedu'] == clase).astype(int)
    drift_cedu_por_clase[clase] = compute_secular_drift(mujeres_fertiles, f'_cedu_es_{clase}', ESTRATOS_DRIFT,
                                                        year_min=ANIO_INICIO_PANEL, year_max=ANIO_FIN_PANEL)
    partner_vector = aplicar_drift_secular_a_grid(partner_vector, f'prob_cedu{clase}', ESTRATOS_DRIFT,
                                                  drift_cedu_por_clase[clase])
partner_vector = renormalizar_multiclase(partner_vector, 'prob_cedu', clases=range(5))
partner_vector.to_parquet(DIR_VEC / 'partner_vector.parquet', engine='pyarrow', compression='snappy', index=False)

[target_couple] holdout 2022-2024: ROC-AUC 0.7688


[target_swork] holdout 2022-2024: ROC-AUC 0.6800


[target_cedu] holdout 2022-2024: log-loss 0.9393


[drift:target_couple] tau^2 2.729e-04 | peso propio medio 57%


[drift:target_swork] tau^2 1.473e-03 | peso propio medio 47%


[drift:_cedu_es_0] tau^2 9.006e-05 | peso propio medio 15%


[drift:_cedu_es_1] tau^2 1.044e-03 | peso propio medio 82%


[drift:_cedu_es_2] tau^2 3.139e-04 | peso propio medio 52%


[drift:_cedu_es_3] tau^2 1.194e-04 | peso propio medio 21%


[drift:_cedu_es_4] tau^2 6.251e-04 | peso propio medio 23%


## Vector de educación: stock (destino educativo por cohorte)

In [8]:
features_stock = ['sex', 'provincia', 'status', 'clima_educativo']
df_edu_stock = people[people['age'] >= 25].copy()
modelo_max_level, metrics_max_level, inv_level = evaluar_modelo(df_edu_stock, features_stock, 'education', num_class=7)

destino_vector = pd.DataFrame(list(itertools.product([0, 1], range(1, 25), range(1, N_ESTRATOS_STATUS + 1),
                                                     range(0, 5))), columns=features_stock)
predecir_multiclase(modelo_max_level, destino_vector[features_stock], inv_level, 7, 'prob_max_level', destino_vector)

ESTRATOS_DRIFT_STOCK = ['sex', 'clima_bucket']
df_edu_stock['cohorte'] = df_edu_stock['year'] - df_edu_stock['age']
drift_stock_por_clase = {}
for clase in range(7):
    df_edu_stock[f'_maxlevel_es_{clase}'] = (df_edu_stock['education'] == clase).astype(int)
    drift_stock_por_clase[clase] = compute_secular_drift(df_edu_stock, f'_maxlevel_es_{clase}',
                                                         ESTRATOS_DRIFT_STOCK, year_col='cohorte')


def aplicar_deriva_cohorte(destino_vector, drift_por_clase, group_cols, tau=TAU_SATURACION, clases=range(7)):
    """Grilla larga por cohorte 1925-2075: deriva lineal hacia atrás y saturante hacia adelante
    de la cohorte ancla, clase por clase, y renormalización al símplex."""
    p_min, p_max = COTAS_PROB['max_level']
    cohortes = np.arange(COHORTE_MIN_VECTOR, COHORTE_MAX_VECTOR + 1)
    base = destino_vector.copy()
    base['clima_bucket'] = base['clima_educativo'].map(BUCKET_CLIMA)
    slopes = {}
    for k in clases:
        d = drift_por_clase[k]
        m = base.merge(d[group_cols + ['slope_final']], on=group_cols, how='left')
        assert len(m) == len(base)
        slopes[k] = m['slope_final'].fillna(d['slope_final'].mean()).to_numpy()
    dt = (cohortes - COHORTE_ANCLA_STOCK).astype(float)
    factor = np.where(dt <= 0, dt, tau * (1.0 - np.exp(-np.maximum(dt, 0.0) / tau)))
    filas = []
    for j, coh in enumerate(cohortes):
        blq = base[features_stock].copy()
        blq['cohorte'] = int(coh)
        P = np.empty((len(base), len(list(clases))), dtype=float)
        for idx, k in enumerate(clases):
            lg = logit(np.clip(base[f'prob_max_level{k}'].to_numpy(), p_min, p_max))
            P[:, idx] = expit(lg + slopes[k] * factor[j])
        P = np.clip(P, p_min, p_max)
        P = P / P.sum(axis=1, keepdims=True)
        for idx, k in enumerate(clases):
            blq[f'prob_max_level{k}'] = P[:, idx].astype(np.float32)
        filas.append(blq)
    return pd.concat(filas, ignore_index=True)


destino_vector_cohorte = aplicar_deriva_cohorte(destino_vector, drift_stock_por_clase, ESTRATOS_DRIFT_STOCK)
assert np.allclose(destino_vector_cohorte[[f'prob_max_level{k}' for k in range(7)]].sum(axis=1), 1.0, atol=1e-5)
destino_vector_cohorte.to_parquet(DIR_VEC / 'education_stock_vector.parquet', engine='pyarrow',
                                  compression='snappy', index=False)

[education] holdout 2022-2024: log-loss 1.1705


[drift:_maxlevel_es_0] tau^2 2.403e-03 | peso propio medio 59%


[drift:_maxlevel_es_1] tau^2 4.640e-05 | peso propio medio 42%


[drift:_maxlevel_es_2] tau^2 2.342e-04 | peso propio medio 68%


[drift:_maxlevel_es_3] tau^2 3.507e-04 | peso propio medio 90%


[drift:_maxlevel_es_4] tau^2 7.382e-04 | peso propio medio 99%


[drift:_maxlevel_es_5] tau^2 2.512e-04 | peso propio medio 98%


[drift:_maxlevel_es_6] tau^2 3.212e-04 | peso propio medio 99%


## Vector de educación: flujo (asistencia condicionada al destino)

In [9]:
features_flujo = ['sex', 'provincia', 'status', 'age', 'clima_educativo', 'max_level_target']
df_edu_flujo = people[people['age'].between(3, 80)].copy()
ESTRATOS_DRIFT_FLUJO = ['tramo_escolar', 'clima_bucket']

# destino observado para quien ya no estudia; imputado con el modelo de stock para quien estudia
df_edu_flujo['max_level_target'] = np.where(df_edu_flujo['target_inschool'] == 0, df_edu_flujo['education'], np.nan)
mask_latente = df_edu_flujo['max_level_target'].isna()
probas = modelo_max_level.predict_proba(df_edu_flujo.loc[mask_latente, features_stock])
P = np.zeros((int(mask_latente.sum()), 7))
for idx_modelo, clase in inv_level.items():
    P[:, int(clase)] = probas[:, int(idx_modelo)]
edu_actual = df_edu_flujo.loc[mask_latente, 'education'].to_numpy()
P[np.arange(7)[None, :] < edu_actual[:, None]] = 0.0
df_edu_flujo.loc[mask_latente, 'max_level_target'] = np.where(P.sum(axis=1) > 0, P.argmax(axis=1), edu_actual)
df_edu_flujo['max_level_target'] = df_edu_flujo['max_level_target'].astype(int)

modelo_inschool, metrics_inschool, _ = evaluar_modelo(df_edu_flujo, features_flujo, 'target_inschool')

grilla_flujo = list(itertools.product([0, 1], range(1, 25), range(1, N_ESTRATOS_STATUS + 1), range(3, 81),
                                      range(0, 5), range(0, 7)))
education_vector = pd.DataFrame(grilla_flujo, columns=features_flujo)
education_vector['tramo_escolar'] = assign_tramo_escolar(education_vector['age'])
education_vector['clima_bucket'] = education_vector['clima_educativo'].map(BUCKET_CLIMA)
education_vector['prob_inschool'] = modelo_inschool.predict_proba(
    education_vector[features_flujo])[:, 1].astype(np.float32)

drift_inschool = compute_secular_drift(df_edu_flujo, 'target_inschool', ESTRATOS_DRIFT_FLUJO,
                                       year_min=ANIO_INICIO_PANEL, year_max=ANIO_FIN_PANEL)
education_vector = aplicar_drift_secular_a_grid(education_vector, 'prob_inschool', ESTRATOS_DRIFT_FLUJO, drift_inschool)
education_vector.to_parquet(DIR_VEC / 'education_flow_vector.parquet', engine='pyarrow',
                            compression='snappy', index=False)

[target_inschool] holdout 2022-2024: ROC-AUC 0.9890


[drift:target_inschool] tau^2 1.447e-03 | peso propio medio 68%


## Diagnósticos: heterogeneidad del *drift* y métricas de validación

In [10]:
drift_tables = {
    "employment_vector (target_work)": drift_work,
    "partner_vector (target_couple)": drift_couple,
    "partner_vector (target_swork)": drift_swork,
    **{f"partner_vector (target_cedu == {c})": drift_cedu_por_clase[c] for c in range(5)},
    "education_flow_vector (target_inschool)": drift_inschool,
    **{f"education_stock_vector (education == {c})": drift_stock_por_clase[c] for c in range(7)},
}
drift_vectores = pd.concat(
    [t.assign(vector=nombre)[["vector", "categoria", "n_anios", "n_obs", "slope", "var_slope", "p_value",
                              "tau2", "peso_propio", "mu_efectos_aleatorios", "slope_final"]]
     for nombre, t in drift_tables.items()], ignore_index=True)
drift_vectores.to_csv(DIR_DIAG / "drift_vectores.csv", index=False)

metricas_vectores = pd.DataFrame([
    dict(metrics_work, vector='employment_vector'), dict(metrics_couple, vector='partner_vector'),
    dict(metrics_swork, vector='partner_vector'), dict(metrics_cedu, vector='partner_vector'),
    dict(metrics_inschool, vector='education_flow_vector'), dict(metrics_max_level, vector='education_stock_vector')])
metricas_vectores = metricas_vectores[['vector', 'target', 'tipo', 'n_clases', 'n_train', 'n_test', 'tasa_base',
                                       'roc_auc', 'pr_auc', 'brier', 'log_loss']]
metricas_vectores.to_csv(DIR_DIAG / 'metricas_vectores.csv', index=False)
metricas_vectores

,vector,target,tipo,n_clases,n_train,n_test,tasa_base,roc_auc,pr_auc,brier,log_loss
0,employment_vector,target_work,binario,2,134034,60984,0.572212,0.794308,0.821942,0.181743,0.541688
1,partner_vector,target_couple,binario,2,134034,60984,0.486294,0.768780,0.694587,0.186814,0.546680
2,partner_vector,target_swork,binario,2,54151,23574,0.942182,0.679994,0.967608,0.053826,0.213628
3,partner_vector,target_cedu,multiclase,5,134034,60984,NaN,NaN,NaN,0.546674,0.939255
4,education_flow_vector,target_inschool,binario,2,479969,217630,0.334534,0.988953,0.982971,0.032813,0.116362
5,education_stock_vector,education,multiclase,7,308942,145021,NaN,NaN,NaN,0.600395,1.170521
